# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
This notebook uses the FAIR^2 dataset, accessible via the following Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Install mlcroissant if not installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display name and description of the dataset
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the dataset's available record sets, fields, and their `@id` identifiers.

We query the dataset to discover its contained record sets. We use `@id` fields for all referencing, as recommended.

In [ ]:
from pprint import pprint

# List all record sets in the dataset using their @id
print("Available Record Sets:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"  - @id: {record_set.id} | name: {record_set.name}")
    record_sets.append(record_set.id)
    
if record_sets:
    # Show fields of the first record set as an example
    print(f"\nFields in the first record set (@id: {record_sets[0]}):")
    record_set_obj = dataset.get_record_set(record_sets[0])
    for field in record_set_obj.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each specific record set into pandas DataFrames for analysis.

Use the record set and field `@id`s from the previous overview.

In [ ]:
dataframes = {}
# We'll iterate through every record set found previously
for record_set_id in record_sets:
    print(f"\nExtracting records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records and isinstance(records[0], dict):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in DataFrame for record set {record_set_id}:")
        print(df.columns.tolist())
        print("Preview:")
        display(df.head())
    else:
        print(f"No tabular records found for this record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by criteria, normalizing numeric fields, categorizing or grouping data, and basic statistics.

*For demonstration, we'll use the first record set (if any) and choose a numeric field (if available).*


In [ ]:
# Example: Select and process numeric fields if available in the first DataFrame
if record_sets:
    main_rs = record_sets[0]
    df_main = dataframes[main_rs]

    # Find numeric columns (int/float) via data type detection
    numeric_cols = df_main.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if not numeric_cols:
        # Alternatively, try to find numeric-type columns from field metadata
        record_set_obj = dataset.get_record_set(main_rs)
        for field in record_set_obj.fields:
            if field.data_type in ('schema:Float', 'schema:Integer', 'schema:Number'):
                if field.id in df_main.columns:
                    numeric_cols.append(field.id)

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field}")
        # Use a threshold value for demonstration
        threshold = df_main[numeric_field].mean() if df_main[numeric_field].dtype != object else 0
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        field_norm = f"{numeric_field}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, field_norm]].head())

        # Attempt to group by the first non-numeric column (categorical)
        group_fields = [c for c in df_main.columns if c != numeric_field and df_main[c].dtype==object]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by field '@id': {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data (mean of {numeric_field} by {group_field}):")
            display(grouped.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields available for EDA in this record set.")
else:
    print("No available record sets for EDA.")

## 5. Visualization
Visualize distributions or relationships in the chosen record set.

*We'll show a histogram of the selected numeric field if available, and a boxplot by a group field if found.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only if we have a numeric field and relevant data
if record_sets and numeric_cols:
    main_rs = record_sets[0]
    df_main = dataframes[main_rs]
    numeric_field = numeric_cols[0]

    plt.figure(figsize=(7,4))
    sns.histplot(df_main[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of '{numeric_field}' (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (categorical) field if possible
    group_fields = [c for c in df_main.columns if c != numeric_field and df_main[c].dtype==object]
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df_main[group_fields[0]], y=df_main[numeric_field])
        plt.title(f"Boxplot of '{numeric_field}' grouped by '{group_fields[0]}' (@id)")
        plt.xlabel(group_fields[0])
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated dataset loading, metadata and schema discovery, record extraction, EDA, and visualization using the `mlcroissant` library.

**Key observations:**
- Used entity `@id` fields to reference all record sets and fields.
- Explored available record sets and their contents dynamically.
- Demonstrated basic EDA and visualizations for fields present in the dataset.

For further analysis, consider deeper statistical modeling or visualization of relationships in the data by leveraging the full structure provided in the Croissant schema.